# 00 · Warehouse tour
The production warehouse (`warehouse.duckdb`) of a food-delivery company. Read-only. 6 tables, Apr–14 Oct 2026. Use this notebook on **Day 1** next to Claude Code.

In [1]:
import pathlib, duckdb
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "warehouse.duckdb").exists())
wh = duckdb.connect(str(ROOT / "warehouse.duckdb"), read_only=True)   # production: ALWAYS read-only
def q(sql, con=None):
    return (con or wh).sql(sql)

## Tables and row counts

In [2]:
q("""SELECT table_name, estimated_size AS approx_rows, column_count
   FROM duckdb_tables() ORDER BY table_name""")

table_nameVARCHAR,approx_rowsBIGINT,column_countBIGINT
cities,6,3
customers,16522,4
order_taxes,169588,2
orders,169588,9
refunds,8460,5
restaurants,237,6


## How the tables connect
```
cities ──< restaurants ──< orders >── customers
                              ├── order_taxes  (GST, 1 row per order)
                              └── refunds      (only refunded orders)
```
An order's **city comes through the restaurant**, not the customer.

In [3]:
q("""SELECT c.city_name, COUNT(*) AS orders
FROM orders o JOIN restaurants r USING (restaurant_id) JOIN cities c USING (city_id)
GROUP BY 1 ORDER BY 2 DESC""")

city_nameVARCHAR,ordersBIGINT
Mumbai,38635
Bengaluru,38423
Delhi,34898
Pune,21530
Hyderabad,21392
Chennai,14710


## Orders per week (spot the Mumbai rain week, 5–11 Oct)

In [4]:
q("""SELECT date_trunc('week', order_ts)::DATE AS week, COUNT(*) AS placed,
       COUNT(*) FILTER (WHERE status = 'cancelled') AS cancelled
FROM orders o JOIN restaurants r USING (restaurant_id)
WHERE r.city_id = 1 AND order_ts >= '2026-09-01'
GROUP BY 1 ORDER BY 1""")

weekDATE,placedBIGINT,cancelledBIGINT
2026-08-31,1296,125
2026-09-07,1420,130
2026-09-14,1445,151
2026-09-21,1447,116
2026-09-28,1422,131
2026-10-05,1311,438
2026-10-12,544,45


## The revenue traps
`amount` includes GST and is before promo; refunded orders still say `delivered`; cancelled orders still have an amount. Compare the naive and correct numbers for September:

In [5]:
q("""SELECT SUM(o.amount) AS naive_sum_amount,
  SUM(o.amount - o.promo_value - t.gst) FILTER (WHERE o.status = 'delivered' AND f.order_id IS NULL) AS revenue
FROM orders o JOIN order_taxes t USING (order_id) LEFT JOIN refunds f USING (order_id)
WHERE o.order_ts >= '2026-09-01' AND o.order_ts < '2026-10-01'""")

"naive_sum_amountDECIMAL(38,2)","revenueDECIMAL(38,2)"
10213435.05,7809309.00


### Exercise
Ask Claude Code: *“Using only warehouse.duckdb, which payment method has the highest refund rate?”* Then verify its SQL here.

In [6]:
# your verification query here
